In [1]:
pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


# Import libraries

In [1]:
import requests
import json
import time
import os
from dotenv import load_dotenv

# Load Twitch credentials

In [9]:
load_dotenv()

True

In [11]:
CLIENT_ID = os.getenv('TWITCH_CLIENT_ID')
CLIENT_SECRET = os.getenv('TWITCH_CLIENT_SECRET')

In [13]:
if not CLIENT_ID or not CLIENT_SECRET:
    print("Error: Credentials missing! Check your local .env file.")

In [15]:
# Ask Twitch for a token before talking to the games database
auth_url = f'https://id.twitch.tv/oauth2/token?client_id={CLIENT_ID}&client_secret={CLIENT_SECRET}&grant_type=client_credentials'
auth_response = requests.post(auth_url)
auth_data = auth_response.json()

In [16]:
# Extract the access token

ACCESS_TOKEN = auth_data.get('access_token')

if ACCESS_TOKEN:
    print("Success! We have a VIP pass.")
else:
    print("Connection failed. Check your ID and Secret.")

Success! We have a VIP pass.


### Test: Handshake Test

Try to find 'Resident Evil' and see its 'Mechanical DNA'

In [19]:
if ACCESS_TOKEN:
    url = "https://api.igdb.com/v4/games"

    headers = {
        'Client-ID': CLIENT_ID,
        'Authorization': f'Bearer {ACCESS_TOKEN}',
    }

    query = 'fields name, genres.name, themes.name, player_perspectives.name; where name = "Resident Evil";'

    response = requests.post(url, headers=headers, data=query)
    print("\n--- ChimeraQuest Test Fetch ---")
    print(json.dumps(response.json(), indent=2))


--- ChimeraQuest Test Fetch ---
[
  {
    "id": 102722,
    "genres": [
      {
        "id": 31,
        "name": "Adventure"
      }
    ],
    "name": "Resident Evil",
    "player_perspectives": [
      {
        "id": 2,
        "name": "Third person"
      }
    ],
    "themes": [
      {
        "id": 19,
        "name": "Horror"
      }
    ]
  },
  {
    "id": 8254,
    "genres": [
      {
        "id": 5,
        "name": "Shooter"
      },
      {
        "id": 9,
        "name": "Puzzle"
      },
      {
        "id": 31,
        "name": "Adventure"
      }
    ],
    "name": "Resident Evil",
    "player_perspectives": [
      {
        "id": 2,
        "name": "Third person"
      }
    ],
    "themes": [
      {
        "id": 1,
        "name": "Action"
      },
      {
        "id": 19,
        "name": "Horror"
      },
      {
        "id": 21,
        "name": "Survival"
      }
    ]
  },
  {
    "id": 24869,
    "genres": [
      {
        "id": 5,
        "name": "Shoo

# The Big Fetch

In [21]:
all_games = []
limit = 500
offset = 0
total_to_fetch = 1500

In [23]:
print(f"Starting ChimeraQuest Data Extraction...")

while offset < total_to_fetch:
    # offset is for getting the next batch of games
    # it is sorted by id to make sure there are no duplicates 

    query = f"""
    fields name, genres.name, themes.name, player_perspectives.name, game_modes.name;
    limit {limit};
    offset {offset};
    where (genres != n & themes != n) & (hypes > 5 | total_rating_count > 10);
    sort id asc;
    """

    response = requests.post(url, headers=headers, data=query)
    batch = response.json()

    if not batch:
        print("No more games found marching the criteria")
        break

    # Error checking to see the response of API
    if isinstance(batch, list):
        if not batch:
            print("No more games found matching the criteria.")
            break
        all_games.extend(batch)
    else:
        print(f"API Error: {batch}")
        break

    if len(all_games) % 500 == 0:
        print(f"Progress Check: Collected {len(all_games)} games so far...")

    offset += limit
    time.sleep(0.25)

Starting ChimeraQuest Data Extraction...
Progress Check: Collected 500 games so far...
Progress Check: Collected 1000 games so far...
Progress Check: Collected 1500 games so far...


In [24]:
# Flattening 
# Turning [{"name": "Horror"}] into just ["Horror"]
flattened_data = []

for game in all_games:
    # returns empty list if field is missing
    get_names = lambda field: [item['name'] for item in game.get(field, [])] if game.get(field) else []
    
    combined_tags = list(set(
        get_names('genres') + 
        get_names('themes') + 
        get_names('player_perspectives') + 
        get_names('game_modes')
    ))

    if combined_tags:
        clean_game = {
            "game_id": game.get('id'),
            "name": game.get('name'),
            "tags": combined_tags
        }
        flattened_data.append(clean_game)

In [25]:
# Save to NDJSON (for BigQuery)
file_name = "chimera_raw_dna.jsonl"
with open(file_name, 'w') as f:
    for entry in flattened_data:
        f.write(json.dumps(entry) + '\n')

print(f"Mission Accomplished!")
print(f"Created {file_name} with {len(flattened_data)} unique game profiles.")

Mission Accomplished!
Created chimera_raw_dna.jsonl with 1500 unique game profiles.
